In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# 1. LOAD DATA
print("Loading data...")
df = pd.read_csv('diabetes.csv')

# 2. INDUSTRIAL CLEANING (Handling the hidden missing values)
# In this dataset, 0 for Glucose, BP, Skin, Insulin, BMI is impossible. It means missing data.
cols_with_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing] = df[cols_with_missing].replace(0, np.nan)

# We use a strategy to fill these gaps based on the median (robust to outliers)
imputer = SimpleImputer(strategy='median')
df[cols_with_missing] = imputer.fit_transform(df[cols_with_missing])

# 3. SPLIT DATA
X = df.drop('Outcome', axis=1)
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. SCALING (Standardizing the range of features)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. DEFINE THE "EXPERTS" (Models)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
xgb = XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
lr = LogisticRegression(random_state=42)

# 6. CREATE THE VOTING ENSEMBLE
# This is the "Industrial" part. We use a 'Soft' voting system to average the probabilities.
model = VotingClassifier(
    estimators=[('rf', rf), ('xgb', xgb), ('lr', lr)],
    voting='soft'
)

# 7. TRAIN
print("Training the ensemble model...")
model.fit(X_train_scaled, y_train)

# 8. EVALUATE
y_pred = model.predict(X_test_scaled)
acc = accuracy_score(y_test, y_pred)
print(f" Model Accuracy: {acc:.2%}")
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 9. SAVE THE BRAIN
# We save the model AND the scaler so the app processes new data exactly like training data
joblib.dump(model, 'diabetes_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print(" Model and Scaler saved successfully!")

Loading data...
Training the ensemble model...
 Model Accuracy: 74.03%

Classification Report:
               precision    recall  f1-score   support

           0       0.80      0.80      0.80        99
           1       0.64      0.64      0.64        55

    accuracy                           0.74       154
   macro avg       0.72      0.72      0.72       154
weighted avg       0.74      0.74      0.74       154

 Model and Scaler saved successfully!


/opt/miniconda3/lib/python3.13/site-packages/xgboost/training.py:199: UserWarning: [21:13:48] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [2]:
import streamlit as st
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Load the trained brain
model = joblib.load('diabetes_model.pkl')
scaler = joblib.load('scaler.pkl')

# --- APP CONFIGURATION ---
st.set_page_config(page_title="Diabetes Risk AI", page_icon="🩺", layout="wide")

# Custom CSS for a "Pro" look
st.markdown("""
    <style>
    .big-font { font-size:30px !important; font-weight: bold; color: #2E86C1; }
    .stButton>button { background-color: #2E86C1; color: white; border-radius: 10px; }
    </style>
    """, unsafe_allow_html=True)

# --- HEADER ---
st.markdown('<p class="big-font">🩺 Intelligent Diabetes Intervention System</p>', unsafe_allow_html=True)
st.markdown("An industrial-grade AI system for early detection and personalized health insights.")
st.divider()

# --- INPUT SECTION (SIDEBAR) ---
st.sidebar.header("Patient Vitals")
def user_input_features():
    pregnancies = st.sidebar.slider('Pregnancies', 0, 17, 3)
    glucose = st.sidebar.slider('Glucose Level (mg/dL)', 0, 200, 117)
    bp = st.sidebar.slider('Blood Pressure (mm Hg)', 0, 122, 72)
    skin = st.sidebar.slider('Skin Thickness (mm)', 0, 99, 23)
    insulin = st.sidebar.slider('Insulin Level (mu U/ml)', 0, 846, 30)
    bmi = st.sidebar.slider('BMI', 0.0, 67.1, 32.0)
    dpf = st.sidebar.slider('Diabetes Pedigree Function', 0.078, 2.42, 0.3725)
    age = st.sidebar.slider('Age (years)', 21, 81, 29)
    
    data = {
        'Pregnancies': pregnancies,
        'Glucose': glucose,
        'BloodPressure': bp,
        'SkinThickness': skin,
        'Insulin': insulin,
        'BMI': bmi,
        'DiabetesPedigreeFunction': dpf,
        'Age': age
    }
    return pd.DataFrame(data, index=[0])

input_df = user_input_features()

# --- MAIN PANEL ---
col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("Patient Data Overview")
    st.dataframe(input_df)

    if st.button('Analyze Risk Profile'):
        # 1. Scale the input using the saved scaler
        input_scaled = scaler.transform(input_df)
        
        # 2. Predict Probability
        prediction_prob = model.predict_proba(input_scaled)
        risk_score = prediction_prob[0][1] # Probability of Class 1 (Diabetes)
        
        # 3. Display Results
        st.subheader("Analysis Results")
        
        # Dynamic Gauge Logic
        if risk_score < 0.3:
            st.success(f"Low Risk: {risk_score:.1%}")
            st.markdown("✅ **Status:** Patient is currently in a healthy range.")
        elif risk_score < 0.7:
            st.warning(f"Moderate Risk: {risk_score:.1%}")
            st.markdown("⚠️ **Status:** Pre-diabetic indicators detected. Lifestyle intervention recommended.")
        else:
            st.error(f"High Risk: {risk_score:.1%}")
            st.markdown("🚨 **Status:** Strong diabetic indicators. Immediate clinical consultation required.")

        # 4. Actionable Insights (The "Winning" Feature)
        st.divider()
        st.subheader("💡 AI-Generated Recommendations")
        if input_df['Glucose'].values[0] > 140:
             st.info("- **Glucose Alert:** Glucose is the primary driver of risk here. Consider testing HbA1c.")
        if input_df['BMI'].values[0] > 30:
             st.info("- **BMI Alert:** Patient is in the obese range. Reducing BMI by 5 points significantly lowers model risk score.")
        if input_df['Age'].values[0] > 50:
             st.info("- **Age Factor:** Age increases baseline risk. Regular screening is critical.")

with col2:
    # A simple visual to show the data relative to population averages
    st.subheader("Vitals vs. Healthy Average")
    # Quick mock-up chart logic
    avg_vals = {'Glucose': 100, 'BP': 70, 'BMI': 22} 
    user_vals = {'Glucose': input_df['Glucose'][0], 'BP': input_df['BloodPressure'][0], 'BMI': input_df['BMI'][0]}
    
    metrics = list(avg_vals.keys())
    
    fig, ax = plt.subplots()
    x = np.arange(len(metrics))
    width = 0.35
    
    rects1 = ax.bar(x - width/2, list(avg_vals.values()), width, label='Healthy Avg', color='green')
    rects2 = ax.bar(x + width/2, list(user_vals.values()), width, label='Patient', color='blue')
    
    ax.set_ylabel('Value')
    ax.set_xticks(x)
    ax.set_xticklabels(metrics)
    ax.legend()
    
    st.pyplot(fig)

2026-02-04 21:13:51.238 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-04 21:13:51.239 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-04 21:13:51.589 
  command:

    streamlit run /opt/miniconda3/lib/python3.13/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-02-04 21:13:51.590 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-04 21:13:51.591 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-04 21:13:51.593 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-04 21:13:51.599 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when ru